# Notebook para entrenar un modelo de ML de precios de casas

Este cuaderno acompaña el taller: entrenamiento del modelo con scikit-learn y despliegue vía API y una app Flask.

In [ ]:
# Configurar dependencias e imports
import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Cargar dataset house_data.csv y verificar columnas
from pathlib import Path
repo_root = Path(__file__).resolve().parents[1]
csv_path = repo_root / 'files' / 'input' / 'house_data.csv'
df = pd.read_csv(csv_path)
required = ['bedrooms','bathrooms','sqft_living','sqft_lot','floors','waterfront','condition']
assert 'price' in df.columns, 'La columna objetivo price no existe'
missing = [c for c in required if c not in df.columns]
print('Faltan columnas:' , missing)
df[required + ['price']].info()
df[required + ['price']].describe().T

In [ ]:
# Definir X e y; dividir en entrenamiento/validación
features = required
df = df.dropna(subset=features + ['price']).copy()
for c in features + ['price']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
X = df[features]
y = df['price']
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
X_train.shape, X_valid.shape

In [ ]:
# Construir Pipeline de preprocesamiento + modelo
num_pre = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
pre = ColumnTransformer([
    ('num', num_pre, features)
])
# Modelo (ajusta a conveniencia):
reg = RandomForestRegressor(random_state=42, n_estimators=300)
pipeline = Pipeline([
    ('pre', pre),
    ('model', reg)
])
pipeline

In [ ]:
# Entrenar, evaluar (RMSE) y registrar métricas
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_valid)
rmse = mean_squared_error(y_valid, y_pred, squared=False)
r2 = r2_score(y_valid, y_pred)
print({"RMSE": rmse, "R2": r2})

In [ ]:
# Guardar el modelo con joblib y metadatos
model_path = Path(__file__).resolve().parent / 'house_predictor.pkl'
joblib.dump({"pipeline": pipeline, "features": features}, model_path)
meta = {
    "features": features,
    "model_path": str(model_path),
}
with open(Path(__file__).resolve().parent / 'model_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
model_path